[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/yy726/GPQA-eval/blob/submission-yy726/notebooks/12_dpo_unsloth.ipynb)

# 12 · DPO on Qwen3 Base（Unsloth + W&B）

结构参考 Unsloth 官方 [Zephyr (7B) DPO](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/Zephyr_(7B)-DPO.ipynb)：
`PatchDPOTrainer()` + TRL `DPOTrainer(ref_model=None)`（LoRA 时参考模型 = 关掉 adapter 的模型）。

**在流程里的位置**（照 OLMo 3：SFT → DPO → RL；OLMo 3 报告：RL 从 DPO 检查点开始比从 SFT 开始更好）：
notebook 11 跑 SFT 并设 `PUSH_SFT_REPO` → 本 notebook 的 `INIT_MODEL` 填它，跑完设 `PUSH_REPO` 推到 HF → 回到 notebook 11，`RUN_SFT=False`、`INIT_MODEL`=这里的 DPO 模型，只跑 GRPO。

**偏好数据**（`DPO_DATA`）：
- `dolci`（默认，**合规地"向 32B 学"**）：OLMo 3 公开的 [`allenai/Dolci-Instruct-DPO`](https://huggingface.co/datasets/allenai/Dolci-Instruct-DPO)，
  其中 `preference_type="delta_learning"` 的对是 **chosen = Qwen3-32B（非思考）、rejected = Qwen3-0.6B** 的回答（[Delta Learning](https://arxiv.org/abs/2507.06187)：
  两边都可以比被训练的模型弱，只要质量差明显，DPO 就能提分，而在 chosen 上直接 SFT 反而有害）。chosen 中位数约 270 token，很简洁。
  按 `prompt_id` 里的来源过滤到科学 / 数学。这是赛前公开的数据集，**可以上榜**。OLMo 3 Instruct：GPQA 30.2 → 34.6–35.5。
- `onpolicy`：MMLU-Pro STEM（或 SciQ）题目，每题让**当前模型**采样 K 个回答；chosen = 答对且正常收尾，rejected = 答错或没停；全对 / 全错的题跳过。
- `both`：两者混合。
- 可选 `allenai/Dolci-Think-DPO-7B`（Qwen3-32B 思考 vs 0.6B 思考，OLMo 3 Think：GPQA 45.1 → 48.4），但科学题 chosen 中位数约 7500 token，长度过滤后剩得少。

`ROUNDS = 2` 时做迭代 DPO：第二轮用第一轮训练后的模型重新采样（参考模型仍是 `INIT_MODEL`）。

| 模式 | SFT 起点 | 采样 | DPO | 评测 | 时间 |
|---|---|---|---|---|---|
| `quick` | 0.6B/1.7B | 200 题 × 4（onpolicy 时） | 200 对，≤1536 token | Diamond 前 60 题 | T4 约 20 分钟 |
| `a100` | 4B | 3000 题 × 8，≤4096 token（onpolicy 时） | 约 8000 对 delta 对，≤4096 token，1 epoch | Diamond + Main，≤8192 | `dolci` 约 40 分钟，`both` 约 1.5 小时（估） |

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv

In [ ]:
# 克隆提交分支（数据准备 / 去污染 / 共用代码 scripts/recipe_utils.py 都在里面）
import os, sys
if not os.path.exists("GPQA-eval"):
    !git clone -q -b submission-yy726 https://github.com/yy726/GPQA-eval.git
%cd GPQA-eval
!git pull -q
sys.path.insert(0, os.path.abspath("scripts"))

In [ ]:
# 安装（约 3–5 分钟）。输出不隐藏：如果安装报错请看这里。
!pip install --upgrade -q uv
try: import numpy, PIL; _numpy = f'numpy=={numpy.__version__}'; _pil = f'pillow=={PIL.__version__}'
except: _numpy = "numpy"; _pil = "pillow"
!uv pip install unsloth vllm wandb {_numpy} {_pil}

In [ ]:
import os, importlib.metadata as md, torch
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"
# T4（计算能力 7.5）不支持 FlashInfer 采样器（A100 不受影响）
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
    print("T4 detected: FlashInfer disabled for vLLM")
GPU_NAME = torch.cuda.get_device_name(0)
GPU_MEM_GB = round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1)
VERSIONS = {}
for p in ["unsloth", "unsloth_zoo", "vllm", "trl", "transformers", "torch", "peft", "wandb"]:
    try: VERSIONS[p] = md.version(p)
    except md.PackageNotFoundError: VERSIONS[p] = None
print(GPU_NAME, GPU_MEM_GB, "GB"); print(VERSIONS)

In [ ]:
# Colab 左侧 🔑 Secrets：HF_TOKEN（必需；先在 https://huggingface.co/datasets/Idavidrein/gpqa 接受条款）
#                      WANDB_API_KEY（可选；没有就不记录 W&B）
for _k in ["HF_TOKEN", "WANDB_API_KEY"]:
    try:
        from google.colab import userdata
        _v = userdata.get(_k)
        if _v:
            os.environ[_k] = _v
    except Exception:
        pass
from huggingface_hub import login
login(token = os.environ.get("HF_TOKEN"))
print("W&B key found" if os.environ.get("WANDB_API_KEY") else "No WANDB_API_KEY: W&B logging disabled")

## 0. 配置

In [ ]:
EXP_NAME     = "D1"
MODE         = "a100"                         # "quick" | "a100"
MODEL_NAME   = "unsloth/Qwen3-4B-Base"        # quick: "unsloth/Qwen3-0.6B-Base"
BASE_MODEL   = "Qwen/Qwen3-4B-Base"
INIT_MODEL   = None     # 推荐：notebook 11 的 PUSH_SFT_REPO，例如 "YOUR_HF_USERNAME/qwen3-4b-x1b-sft"
PUSH_REPO    = None     # 例如 "YOUR_HF_USERNAME/qwen3-4b-d1-dpo"：DPO 后合并推到 HF（notebook 11 的 GRPO 从它开始）
LOAD_IN_4BIT = False
CHAT_STYLE   = "olmo"   # 和 notebook 11 一致：回答以 <|endoftext|> 结束

DPO_DATA         = "dolci"           # "dolci"（Dolci-Instruct-DPO 的 32B vs 0.6B delta 对）| "onpolicy" | "both"
PROMPT_SOURCE    = "mmlu_pro_stem"   # 在线偏好对的题库："mmlu_pro_stem" | "sciq"
ROUNDS           = 1                 # 2 = 迭代 DPO
PAIRS_PER_PROMPT = 1                 # 每题最多几对（a100 可试 2）

DPO_BETA  = 0.1
DPO_LR    = 5e-6
LOSS_TYPE = "sigmoid"                # 标准 DPO；"ipo" = 按长度平均 logp（OLMo 3 用的是长度归一化的 dpo_norm，TRL 没有现成实现）
RPO_ALPHA = None                     # chosen 上加 NLL 项（≈ 部分 SFT）。delta learning 不要在 chosen 上 SFT，所以默认关；onpolicy 时可试 1.0

DOLCI_DPO_DATASET = "allenai/Dolci-Instruct-DPO"   # 或 "allenai/Dolci-Think-DPO-7B"（长，过滤后剩得少）
DOLCI_DPO_TYPES   = ["delta_learning"]             # None = 也用 LLM 判分的对
DOLCI_DPO_INCLUDE = ["science", "sciriff", "math", "gsm"]   # 匹配 prompt_id / dataset_source
DOLCI_DPO_EXCLUDE = ["safety", "wildguard", "wildjailbreak", "python", "code", "aya", "precise", "flan"]

WANDB_PROJECT = os.environ.get("WANDB_PROJECT", "gpqa-hackathon")
WANDB_ENTITY  = os.environ.get("WANDB_ENTITY")

PRESETS = dict(
    quick = dict(max_seq_len=2048, gpu_util=0.6, lora_rank=16, n_prompts=200, k=4, gen_max=512,
                 dolci_n=200, dpo_max_len=1536, dpo_batch=2, dpo_accum=4,
                 eval_configs=["gpqa_diamond"], eval_n=60, eval_max_tokens=1024, shuffle_buffer=1000),
    a100 = dict(max_seq_len=12288, gpu_util=0.6, lora_rank=64, n_prompts=3000, k=8, gen_max=4096,
                dolci_n=8000, dpo_max_len=4096, dpo_batch=2, dpo_accum=8,
                eval_configs=["gpqa_diamond", "gpqa_main"], eval_n=None, eval_max_tokens=8192, shuffle_buffer=20000),
)
Q = PRESETS[MODE]
MAX_SEQ_LEN, GPU_UTIL, LORA_RANK = Q["max_seq_len"], Q["gpu_util"], Q["lora_rank"]
if INIT_MODEL is None:
    print("[!] INIT_MODEL 为空：直接从 base 做 DPO。建议先跑 notebook 11（设 PUSH_SFT_REPO），再把它填到这里。")
print(EXP_NAME, MODE, INIT_MODEL or MODEL_NAME, Q)

## 1. 加载模型（`fast_inference=True`：采样、评测、训练共用一个 vLLM 引擎）

In [ ]:
import os, torch, json, random, time
if torch.cuda.get_device_capability()[0] < 8:
    os.environ["UNSLOTH_VLLM_NO_FLASHINFER"] = "1"
    os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"
from unsloth import FastLanguageModel
try:                                   # 官方 DPO 示例要求先打补丁；新版 Unsloth 可能已自动处理
    from unsloth import PatchDPOTrainer
    PatchDPOTrainer()
except ImportError:
    print("[i] PatchDPOTrainer not available in this Unsloth version (trainers are patched automatically)")
from recipe_utils import (STOP_STRS, setup_chat, chat_prompt, build_gpqa, score_vllm_outputs, log_eval, dolci_dpo_pair,
                          load_mcq_rows, is_correct_and_stopped, decontam_rows, n_chat_tokens, config_kwargs,
                          wandb_init, print_table)

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = INIT_MODEL or MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit = LOAD_IN_4BIT,
    fast_inference = True,
    max_lora_rank = LORA_RANK,
    gpu_memory_utilization = GPU_UTIL,
)
model = FastLanguageModel.get_peft_model(
    model,
    r = LORA_RANK,
    target_modules = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_alpha = LORA_RANK * 2,
    use_gradient_checkpointing = "unsloth",
    random_state = 3407,
)
STOP_IDS = setup_chat(tokenizer, model, style = CHAT_STYLE)   # 回答以 eos=<|endoftext|> 结束（照 OLMo 3）
print("eos:", tokenizer.eos_token, "| stop ids:", STOP_IDS, "| pad:", tokenizer.pad_token)

RUN_NAME = f"{EXP_NAME}-dpo-{MODEL_NAME.split('/')[-1]}-{MODE}"
wandb, REPORT_TO = wandb_init(WANDB_PROJECT, RUN_NAME, dict(
    exp = EXP_NAME, recipe = "dpo", mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL, base_model = BASE_MODEL,
    dpo_data = DPO_DATA, prompt_source = PROMPT_SOURCE, rounds = ROUNDS, pairs_per_prompt = PAIRS_PER_PROMPT,
    chat_style = CHAT_STYLE, dolci_dpo = DOLCI_DPO_DATASET, dolci_dpo_types = DOLCI_DPO_TYPES, dolci_dpo_include = DOLCI_DPO_INCLUDE,
    beta = DPO_BETA, lr = DPO_LR, loss_type = LOSS_TYPE, rpo_alpha = RPO_ALPHA,
    gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB, versions = VERSIONS, **Q), WANDB_ENTITY)
REPORT = dict(notebook = "12_dpo_unsloth", exp = EXP_NAME, mode = MODE, model = MODEL_NAME, init_model = INIT_MODEL,
              dpo_data = DPO_DATA, prompt_source = PROMPT_SOURCE, rounds = ROUNDS, beta = DPO_BETA, lr = DPO_LR,
              loss_type = LOSS_TYPE, rpo_alpha = RPO_ALPHA, q = Q)

## GPQA 快速评测（只输出聚合数字；不记录题目或模型输出）

口径和 notebook 11 完全一样（`scripts/recipe_utils.py`）：`flex` = 输出中最后一个 `(A)`–`(D)`（排行榜指标），`strict` = 第一个 `The answer is (X)`。
⚠️ 不是排行榜分数（单样本、选项顺序与 lm-eval 不同），只用来比较同一次运行里训练前后的变化。

In [ ]:
from vllm import SamplingParams

EVAL_SETS = {}
for cfg in Q["eval_configs"]:
    rows = build_gpqa(cfg, Q["eval_n"])
    texts = [chat_prompt(tokenizer, p) for p, _ in rows]
    longest = max(len(tokenizer(t, add_special_tokens = False)["input_ids"]) for t in texts)
    EVAL_SETS[cfg] = (texts, [g for _, g in rows], longest)
EVAL_MAX_TOKENS = min(Q["eval_max_tokens"], MAX_SEQ_LEN - max(v[2] for v in EVAL_SETS.values()) - 8)
RESULTS = {}

def quick_eval(tag, lora_request = None):
    sp = SamplingParams(temperature = 0.6, top_p = 0.95, max_tokens = EVAL_MAX_TOKENS, seed = 0,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    RESULTS[tag] = {}
    for cfg, (texts, golds, _) in EVAL_SETS.items():
        t0 = time.time()
        r = score_vllm_outputs(model.fast_generate(texts, sampling_params = sp, lora_request = lora_request), golds)
        r["minutes"] = round((time.time() - t0) / 60, 1)
        RESULTS[tag][cfg] = r
        print(tag, cfg, r)
    log_eval(wandb, RESULTS[tag], tag)

quick_eval("0_init" if INIT_MODEL else "0_base")

## 2. 偏好数据

- 在线偏好对的题目先过 GPQA 13-gram 去污染，再按长度过滤（题目 + 回答上限 ≤ `dpo_max_len`）。
- Dolci DPO：流式读取，按来源 / domain 过滤，`chosen` / `rejected` 两段都要 ≤ `dpo_max_len`，同样去污染。

In [ ]:
from datasets import load_dataset

DATA_INFO = {}
mcq = []
if DPO_DATA in ("onpolicy", "both"):
    # SciQ 前 6000 条可能已被 notebook 11 用于 SFT / GRPO，跳过
    mcq = load_mcq_rows(PROMPT_SOURCE, Q["n_prompts"], skip = 6000 if PROMPT_SOURCE == "sciq" else 0)
    mcq, DATA_INFO["onpolicy_dropped_decontam"] = decontam_rows(mcq, lambda r: {"prompt": r["prompt"]}, "dpo_prompts")
    budget = Q["dpo_max_len"] - Q["gen_max"]
    mcq = [r for r in mcq if n_chat_tokens(tokenizer, [{"role": "user", "content": r["prompt"]}], True) <= budget]
    DATA_INFO["onpolicy_prompts"] = len(mcq)

def load_dolci_dpo(n):
    ds = load_dataset(DOLCI_DPO_DATASET, split = "train", streaming = True).shuffle(seed = 0, buffer_size = Q["shuffle_buffer"])
    out, scanned = [], 0
    for ex in ds:
        if len(out) >= n or scanned >= 300_000:
            break
        if scanned == 0:
            print(DOLCI_DPO_DATASET, "columns:", sorted(ex.keys()))
        scanned += 1
        pair = dolci_dpo_pair(ex, DOLCI_DPO_INCLUDE, DOLCI_DPO_EXCLUDE, DOLCI_DPO_TYPES)
        if pair is None:
            continue
        lc = n_chat_tokens(tokenizer, pair["prompt"] + pair["chosen"])
        lr = n_chat_tokens(tokenizer, pair["prompt"] + pair["rejected"])
        if max(lc, lr) > Q["dpo_max_len"]:
            continue
        pair.update(chosen_tokens = lc, rejected_tokens = lr)
        out.append(pair)
    print(f"Dolci DPO: kept {len(out)} of {scanned} scanned")
    return out

dolci_pairs = []
if DPO_DATA in ("dolci", "both"):
    dolci_pairs = load_dolci_dpo(Q["dolci_n"])
    dolci_pairs, DATA_INFO["dolci_dropped_decontam"] = decontam_rows(
        dolci_pairs, lambda r: {"prompt": r["prompt"], "completion": r["chosen"], "text": r["rejected"][0]["content"]},
        "dpo_dolci")
    from collections import Counter
    DATA_INFO["dolci_pairs"] = len(dolci_pairs)
    DATA_INFO["dolci_sources"] = Counter(r["source"].rsplit("-request", 1)[0] for r in dolci_pairs).most_common(10)
    DATA_INFO["dolci_models"] = Counter(f'{r["chosen_model"]} > {r["rejected_model"]}' for r in dolci_pairs).most_common(5)
    if dolci_pairs:   # 长度差：防止模型靠"写长"投机（OLMo 3 用长度归一化的 DPO 也是为此）
        DATA_INFO["dolci_mean_tokens"] = dict(chosen = round(sum(r["chosen_tokens"] for r in dolci_pairs) / len(dolci_pairs)),
                                              rejected = round(sum(r["rejected_tokens"] for r in dolci_pairs) / len(dolci_pairs)))
print(DATA_INFO)

In [ ]:
def make_pairs(rows, lora_request, rnd):
    # 每题采样 K 个回答 -> (答对且收尾, 其余) 配对。只打印聚合数字（MMLU-Pro / SciQ 是公开数据，但也不必打印）
    sp = SamplingParams(n = Q["k"], temperature = 1.0, top_p = 1.0, max_tokens = Q["gen_max"], seed = 3407 + rnd,
                        stop_token_ids = STOP_IDS, stop = STOP_STRS)
    t0 = time.time()
    outs = model.fast_generate([chat_prompt(tokenizer, r["prompt"]) for r in rows], sampling_params = sp,
                               lora_request = lora_request)
    rng = random.Random(rnd)
    pairs, n_good, n_trunc, n_all, all_good, all_bad = [], 0, 0, 0, 0, 0
    for r, o in zip(rows, outs):
        good = [c for c in o.outputs if is_correct_and_stopped(c.text, r["answer"], c.finish_reason)]
        bad = [c for c in o.outputs if c not in good]
        n_good += len(good); n_all += len(o.outputs)
        n_trunc += sum(c.finish_reason == "length" for c in o.outputs)
        if not good:
            all_bad += 1; continue
        if not bad:
            all_good += 1; continue
        for _ in range(min(PAIRS_PER_PROMPT, len(good), len(bad))):
            ch, rj = good.pop(rng.randrange(len(good))), bad.pop(rng.randrange(len(bad)))
            pairs.append({"prompt": [{"role": "user", "content": r["prompt"]}],
                          "chosen": [{"role": "assistant", "content": ch.text}],
                          "rejected": [{"role": "assistant", "content": rj.text}],
                          "chosen_tokens": len(ch.token_ids), "rejected_tokens": len(rj.token_ids),
                          "rejected_truncated": rj.finish_reason == "length"})
    n = max(1, len(rows))
    info = dict(round = rnd, prompts = len(rows), pairs = len(pairs),
                sample_correct_and_stopped = round(100 * n_good / max(1, n_all), 1),
                sample_truncation_rate = round(100 * n_trunc / max(1, n_all), 1),
                frac_all_good = round(100 * all_good / n, 1), frac_all_bad = round(100 * all_bad / n, 1),
                mean_chosen_tokens = round(sum(p["chosen_tokens"] for p in pairs) / max(1, len(pairs))),
                mean_rejected_tokens = round(sum(p["rejected_tokens"] for p in pairs) / max(1, len(pairs))),
                rejected_truncated = round(100 * sum(p["rejected_truncated"] for p in pairs) / max(1, len(pairs)), 1),
                minutes = round((time.time() - t0) / 60, 1))
    print(info)
    if wandb.run is not None:
        for k, v in info.items():
            wandb.run.summary[f"pairs/r{rnd}/{k}"] = v
    return pairs, info

## 3. DPO 训练

`ref_model=None`：LoRA 训练时 TRL 用"关掉 adapter 的模型"当参考模型，也就是 `INIT_MODEL`，不用再占一份显存。
`config_kwargs` 只保留当前 TRL 版本的 `DPOConfig` 真有的参数（例如新版本删掉了某个字段，会打印"ignored"而不是报错）。

In [ ]:
import numpy as np
from datasets import Dataset
from trl import DPOTrainer, DPOConfig

def _avg(log, key, part):
    vals = [h[key] for h in log if key in h]
    k = max(1, len(vals) // 5)
    return round(float(np.mean(vals[:k] if part == "first" else vals[-k:])), 4) if vals else None

ROUND_INFO, DPO_STATS, lora_req = [], [], None
for rnd in range(ROUNDS):
    train_rows = []
    if DPO_DATA in ("onpolicy", "both"):
        pairs, info = make_pairs(mcq, lora_req, rnd)
        ROUND_INFO.append(info)
        train_rows += pairs
    if DPO_DATA in ("dolci", "both") and rnd == 0:
        train_rows += dolci_pairs
    if not train_rows:
        print("[!] 没有可用的偏好对（全对或全错？）：换题库或检查 INIT_MODEL"); break
    random.Random(rnd).shuffle(train_rows)
    ds = Dataset.from_list([{k: r[k] for k in ("prompt", "chosen", "rejected")} for r in train_rows])

    extra = {"rpo_alpha": RPO_ALPHA} if RPO_ALPHA is not None else {}
    args = DPOConfig(**config_kwargs(
        DPOConfig,
        beta = DPO_BETA, loss_type = LOSS_TYPE, learning_rate = DPO_LR,
        per_device_train_batch_size = Q["dpo_batch"], gradient_accumulation_steps = Q["dpo_accum"],
        num_train_epochs = 1, warmup_ratio = 0.1, lr_scheduler_type = "linear", optim = "adamw_8bit",
        weight_decay = 0.0, logging_steps = 5, max_length = Q["dpo_max_len"], max_prompt_length = 1024,
        seed = 3407, report_to = REPORT_TO, run_name = f"{RUN_NAME}-r{rnd}", output_dir = f"outputs/dpo_r{rnd}",
        save_strategy = "no", **extra))
    trainer = DPOTrainer(model = model, ref_model = None, args = args, train_dataset = ds, processing_class = tokenizer)
    stats = trainer.train()
    log = trainer.state.log_history
    s = dict(round = rnd, pairs = len(ds), steps = trainer.state.global_step,
             minutes = round(stats.metrics["train_runtime"] / 60, 1))
    for key in ["loss", "rewards/margins", "rewards/accuracies", "rewards/chosen", "rewards/rejected",
                "logps/chosen", "logps/rejected", "nll_loss"]:
        if any(key in h for h in log):
            s[key] = {"first": _avg(log, key, "first"), "last": _avg(log, key, "last")}
    DPO_STATS.append(s)
    print(s)
    model.save_lora(f"dpo_lora_r{rnd}")
    lora_req = model.load_lora(f"dpo_lora_r{rnd}")
    quick_eval(f"{rnd + 1}_dpo_r{rnd}", lora_req)
    del trainer, ds
    torch.cuda.empty_cache()
REPORT.update(data = DATA_INFO, pairs = ROUND_INFO, dpo = DPO_STATS)

## 4. 汇总（W&B 上也有同样的表；最后打印的 JSON 可以贴回给 Claude）

In [ ]:
rows = print_table(RESULTS)
REPORT.update(quick_eval = RESULTS, versions = VERSIONS, gpu = GPU_NAME, gpu_mem_gb = GPU_MEM_GB,
              wandb = wandb.run.url if REPORT_TO == "wandb" else None)
os.makedirs("outputs", exist_ok = True)
json.dump(REPORT, open(f"outputs/report_{EXP_NAME}.json", "w"), indent = 1, default = str)
if REPORT_TO == "wandb":
    wandb.log({"gpqa_quick_eval": wandb.Table(columns = ["stage", "set", "acc_flex", "acc_strict", "extract_rate",
                                                          "truncation_rate", "mean_tokens"], data = rows)})
    art = wandb.Artifact(f"report-{RUN_NAME}", type = "report")
    art.add_file(f"outputs/report_{EXP_NAME}.json")
    wandb.log_artifact(art)
    wandb.finish()
print("=" * 30, "把下面整段 JSON 贴回给 Claude", "=" * 30)
print(json.dumps(REPORT, ensure_ascii = False, default = str))

## 5. 合并保存 → 推到 Hub（下一步 GRPO 从这里开始）

`merged_16bit` 把 LoRA 合并进权重；保存的 tokenizer 带着模板和 eos=`<|endoftext|>`。设了 `PUSH_REPO` 就推到 HF 私有仓库，
然后在 notebook 11 里 `RUN_SFT=False`、`INIT_MODEL=PUSH_REPO` 跑 GRPO。正式评测用 `main` 的 `01`（生成预算 8192）；A100 上训练的模型提交时注明 A100（不参与排名）。

In [ ]:
if PUSH_REPO:
    try:
        model.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", private = True,
                                 token = os.environ.get("HF_TOKEN"))
    except TypeError:
        model.push_to_hub_merged(PUSH_REPO, tokenizer, save_method = "merged_16bit", token = os.environ.get("HF_TOKEN"))
    print(f"pushed: https://huggingface.co/{PUSH_REPO} → notebook 11: RUN_SFT=False, INIT_MODEL = PUSH_REPO")
else:
    model.save_pretrained_merged("outputs/merged_16bit", tokenizer, save_method = "merged_16bit")
    print("saved locally: outputs/merged_16bit（设 PUSH_REPO 可以直接推到 HF）")